## Transformaciones: Artefactos y uso

Se construyen los tres artefactos que alimentan las secciones subsiguientes

### Constantes

- `MAX_PART`: máximo de participantes de un proceso para generar pares de co-participación

- `MIN_COMP`: procesos compartidos mínimos para crear una arista de co-participación

In [1]:
MAX_PART = 30
MIN_COMP = 2

### Utilidades

In [2]:
NOTEBOOK = "c_uso"
%run ../funciones.ipynb

In [3]:
spark = crear_spark(NOTEBOOK)
verificar_cluster(spark)

Master        : spark://spark-master:7077
Executors     : 2
Cores totales : 2
Memoria driver: 2.0 GB
UI del job    : http://localhost:4040


### Carga

In [4]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

hechos = leer(final_dir, "hechos").where(F.col("anio").isin(anios))
proc   = leer(final_dir, "procesos").where(F.col("anio").isin(anios))
part   = leer(final_dir, "participantes").where(F.col("anio").isin(anios))
dim_e, dim_p = leer(final_dir, "entidades"), leer(final_dir, "proveedores")

hp = (hechos.groupBy("ocid", "award_id", "proveedor_key", "entidad_key", "region", "categoria",
                     "postor_unico", "es_directa", "anio", "mes")
            .agg(F.sum("monto_asignado").alias("monto"))).cache()
print(f"award × proveedor: {hp.count():,}")

award × proveedor: 199,377


---
### **1. Vectores de características**

#### a. Proveedor

In [5]:
def dependencia(df, actor, contraparte, sufijo):
    w = Window.partitionBy(actor)
    return (df.groupBy(actor, contraparte).agg(F.sum("monto").alias("m"))
              .withColumn("s", F.col("m") / F.sum("m").over(w))
              .groupBy(actor).agg(F.max("s").alias(f"share_top_{sufijo}"), F.sum(F.col("s") ** 2).alias(f"hhi_{sufijo}"),
                                  F.count("*").alias(f"n_{sufijo}")))

cat = lambda v: F.avg((F.col("categoria") == v).cast("int"))
n_familias = lambda k: hechos.where("familia <> 'SIN'").groupBy(k).agg(F.countDistinct("familia").alias("n_familias"))

In [6]:
ofe_p = part.groupBy("proveedor_key").agg(F.sum("en_lista_oferentes").alias("n_ofertas"),
                                          F.sum(F.col("en_lista_oferentes") * F.col("gano")).alias("n_ganadas_compitiendo"))
feat_prov = (hp.groupBy("proveedor_key").agg(
        F.countDistinct("ocid").alias("n_procesos"), F.countDistinct("region").alias("n_regiones"),
        F.sum("monto").alias("monto_total"), F.expr("percentile_approx(monto, 0.5)").alias("monto_mediano"),
        F.avg("postor_unico").alias("pct_postor_unico"), F.avg("es_directa").alias("pct_directa"),
        cat("GOODS").alias("pct_bienes"), cat("SERVICES").alias("pct_servicios"), cat("WORKS").alias("pct_obras"),
        F.countDistinct("anio", "mes").alias("meses_activos"))
    .join(dependencia(hp, "proveedor_key", "entidad_key", "entidades"), "proveedor_key", "left")
    .join(n_familias("proveedor_key"), "proveedor_key", "left").join(ofe_p, "proveedor_key", "left")
    .join(dim_p.select("proveedor_key", "es_consorcio"), "proveedor_key", "left")
    .withColumn("compite", (F.coalesce("n_ofertas", F.lit(0)) > 0).cast("int"))
    .withColumn("tasa_exito", F.when(F.col("n_ofertas") > 0, F.col("n_ganadas_compitiendo") / F.col("n_ofertas")))
    .withColumn("log_n_procesos", F.log1p("n_procesos"))
    .withColumn("log_monto_total", F.log1p("monto_total")).withColumn("log_monto_mediano", F.log1p("monto_mediano")))

In [7]:
FEAT_PROV = ["log_n_procesos", "n_entidades", "n_regiones", "log_monto_total", "log_monto_mediano", "pct_postor_unico",
             "pct_directa", "pct_bienes", "pct_servicios", "pct_obras", "meses_activos", "share_top_entidades",
             "hhi_entidades", "n_familias", "compite", "tasa_exito", "es_consorcio"]
feat_prov = VectorAssembler(inputCols=FEAT_PROV, outputCol="features_raw").transform(feat_prov.fillna(0, FEAT_PROV))

#### b. Entidad

Las métricas de proceso salen de **todos** sus procesos

In [8]:
adjudicados = hp.select("ocid").distinct().withColumn("adjudicado", F.lit(1))
feat_ent = (proc.join(adjudicados, "ocid", "left").fillna({"adjudicado": 0})
    .groupBy("entidad_key").agg(
        F.count("*").alias("n_procesos"), F.avg("postor_unico").alias("pct_postor_unico"),
        F.avg("es_directa").alias("pct_directa"), F.avg("n_postores").alias("postores_promedio"),
        F.avg(1 - F.col("adjudicado")).alias("pct_sin_adjudicar"),
        cat("GOODS").alias("pct_bienes"), cat("SERVICES").alias("pct_servicios"), cat("WORKS").alias("pct_obras"))
    .join(hp.groupBy("entidad_key").agg(F.sum("monto").alias("monto_adjudicado_total"),
                                        F.expr("percentile_approx(monto, 0.5)").alias("monto_mediano")), "entidad_key", "left")
    .join(dependencia(hp, "entidad_key", "proveedor_key", "proveedores"), "entidad_key", "left")
    .join(n_familias("entidad_key"), "entidad_key", "left")
    .join(dim_e.select("entidad_key", "region"), "entidad_key", "left")
    .withColumn("log_n_procesos", F.log1p("n_procesos"))
    .withColumn("log_monto_total", F.log1p("monto_adjudicado_total")).withColumn("log_monto_mediano", F.log1p("monto_mediano")))

FEAT_ENT = ["log_n_procesos", "pct_postor_unico", "pct_directa", "postores_promedio", "pct_sin_adjudicar", "pct_bienes",
            "pct_servicios", "pct_obras", "log_monto_total", "log_monto_mediano", "share_top_proveedores",
            "hhi_proveedores", "n_proveedores", "n_familias"]
feat_ent = VectorAssembler(inputCols=FEAT_ENT, outputCol="features_raw").transform(feat_ent.fillna(0, FEAT_ENT))

`features_proveedor` (86,017 × 17 features)

| proveedor_key | log_n_procesos | n_entidades | hhi_entidades | tasa_exito | es_consorcio | … | features_raw |
|---|---|---|---|---|---|---|---|
| `RUC:20511037001` | 7.44 | 275 | 0.04 | 0.44 | 0 | … | [7.44, 275, …] |
| `CONS:1756338` | 0.69 | 1 | 1.00 | 1.00 | 1 | … | [0.69, 1, …] |

`features_entidad` (2,996 × 14 features)

| entidad_key | log_n_procesos | n_proveedores | hhi_proveedores | pct_postor_unico | pct_sin_adjudicar | … | features_raw |
|---|---|---|---|---|---|---|---|
| `ENT:2543` | 8.63 | 1,917 | 0.01 | 0.12 | 0.27 | … | [8.63, 0.12, …] |
| `ENT:723` | 3.40 | 17 | 0.19 | 0.15 | 0.38 | … | [3.40, 0.15, …] |


- **Uso:** KMeans++, DBSCAN/HDBSCAN y jerárquico; tipologías de actores y outliers

- `log1p` en montos y conteos por la cola pesada

- `share_top_*` y `hhi_*` miden dependencia: HHI = Σ participación², 1 = una sola contraparte

- `compite` separa "nunca oferta" de "oferta y nunca gana"

- `es_consorcio`: el consorcio es un nodo propio, porque la fuente no publica sus miembros

---

### **2. Matrices de interacción**

Una fila por celda no vacía, con índices enteros para ALS y SVD, y **una fila por año**

In [9]:
def indexar(df, entrada, salida):
    idx = StringIndexer(inputCol=entrada, outputCol=salida).fit(df)
    return idx, idx.labelsArray[0]

def a_enteros(df, cols):
    for c in cols:
        df = df.withColumn(c, F.col(c).cast("int"))
    return df

ep = (hp.groupBy("entidad_key", "proveedor_key", "anio")
        .agg(F.count("*").alias("n_adj"), F.sum("monto").alias("monto")))
ef = (hechos.where("familia <> 'SIN'").groupBy("entidad_key", "familia", "anio")
            .agg(F.countDistinct("ocid").alias("n_procesos"), F.sum("monto_asignado").alias("monto")))
idx_e, lab_e = indexar(ep.select("entidad_key").unionByName(ef.select("entidad_key")), "entidad_key", "entidad_idx")
idx_p, lab_p = indexar(ep, "proveedor_key", "proveedor_idx")
idx_f, lab_f = indexar(ef, "familia", "familia_idx")

In [10]:
m_ep = (a_enteros(idx_p.transform(idx_e.transform(ep)), ["entidad_idx", "proveedor_idx"])
          .withColumn("senal_log_monto", F.log1p(F.coalesce("monto", F.lit(0)))))
m_ef = a_enteros(idx_f.transform(idx_e.transform(ef)), ["entidad_idx", "familia_idx"])
indices = {k: spark.createDataFrame(list(enumerate(v)), ["idx", "clave"])
           for k, v in {"indice_entidad": lab_e, "indice_proveedor": lab_p, "indice_familia": lab_f}.items()}

def densidad(m, fila, columna, n_f, n_c):
    nnz = m.select(fila, columna).distinct().count()
    return nnz, nnz / (n_f * n_c)

n_e, n_p, n_f = len(lab_e), len(lab_p), len(lab_f)
(nnz_ep, d_ep), (nnz_ef, d_ef) = densidad(m_ep, "entidad_idx", "proveedor_idx", n_e, n_p), densidad(m_ef, "entidad_idx", "familia_idx", n_e, n_f)
print(f"entidad × proveedor: {n_e:,} × {n_p:,} | pares {nnz_ep:,} | densidad {d_ep:.4%}")
print(f"entidad × familia  : {n_e:,} × {n_f:,} | pares {nnz_ef:,} | densidad {d_ef:.2%}")

entidad × proveedor: 2,992 × 86,017 | pares 136,099 | densidad 0.0529%
entidad × familia  : 2,992 × 386 | pares 39,348 | densidad 3.41%


`matriz_entidad_proveedor`

| entidad_idx | proveedor_idx | anio | n_adj | monto | senal_log_monto |
|---|---|---|---|---|---|
| 0 | 1 | 2023 | 2 | 109.4M | 18.5 |
| 0 | 1 | 2024 | 1 | 60.7k | 11.0 |

`matriz_entidad_familia`

| entidad_idx | familia_idx | anio | n_procesos | monto |
|---|---|---|---|---|
| 0 | 1 | 2023 | 17 | 19.5M |
| 0 | 1 | 2024 | 19 | 32.9M |

- `matriz_entidad_proveedor`: ALS implícito y filtrado por vecindario (¿a quién le compra?)

- `matriz_entidad_familia`: perfil de rubros; *cold start* y para distinguir sustituto de rubro nuevo

- **Uso:** SVD truncada de la matriz entidad × proveedor como embedding de entidades y proveedores



---
### **3. Grafos**

#### a. Bipartito entidad → proveedor

In [11]:
e_p = hp.groupBy("entidad_key", "proveedor_key").agg(F.count("*").alias("n_adj"), F.sum("monto").alias("monto"),
                                                    F.min("anio").alias("anio_min"), F.max("anio").alias("anio_max"))
nodo = lambda df, k, pref, tipo, cols: df.select(F.concat(F.lit(pref), k).alias("id"), F.lit(tipo).alias("tipo"), *cols)
v_ent = nodo(dim_e.join(e_p.select("entidad_key").distinct(), "entidad_key"), "entidad_key", "E|", "entidad", ["nombre", "region"])
v_prov = nodo(dim_p.join(e_p.select("proveedor_key").distinct(), "proveedor_key"), "proveedor_key", "P|", "proveedor",
              ["nombre", "region", "es_consorcio"])
vertices_bip = v_ent.unionByName(v_prov, allowMissingColumns=True)
aristas_bip = e_p.select(F.concat(F.lit("E|"), "entidad_key").alias("src"), F.concat(F.lit("P|"), "proveedor_key").alias("dst"),
                         "n_adj", "monto", "anio_min", "anio_max")

- **Nodos:** entidades (`E|`) y proveedores (`P|`); el prefijo evita fundir una entidad que también es proveedor

- **Arista dirigida** entidad → proveedor si hubo al menos una adjudicación; peso `n_adj` o `monto`

- **Uso:** PageRank (GraphFrames) e HITS

- Hubs = entidades que compran a proveedores centrales

- Authorities =  proveedores a los que compran buenos hubs (PQ1)

#### b. Co-participación proveedor — proveedor

In [12]:
SHUFFLE = int(spark.conf.get("spark.sql.shuffle.partitions"))

n_part = part.groupBy("ocid").agg(F.count("*").alias("n"))
part_n = (part.select("ocid", "proveedor_key").join(n_part.where(f"n BETWEEN 2 AND {MAX_PART}"), "ocid")
              .repartition(SHUFFLE, "ocid").cache())
pares = (part_n.alias("a").join(part_n.alias("b"),
            (F.col("a.ocid") == F.col("b.ocid")) & (F.col("a.proveedor_key") < F.col("b.proveedor_key")))
         .select(F.col("a.proveedor_key").alias("src"), F.col("b.proveedor_key").alias("dst"), F.col("a.n").alias("n")))
aristas_co = (pares.groupBy("src", "dst").agg(F.count("*").alias("n_coparticipa"), F.sum(1.0 / (F.col("n") - 1)).alias("peso"))
              .where(f"n_coparticipa >= {MIN_COMP}")
              .withColumn("src", F.concat(F.lit("P|"), "src")).withColumn("dst", F.concat(F.lit("P|"), "dst"))).cache()

In [13]:
extremos = aristas_co.select(F.col("src").alias("id")).unionByName(aristas_co.select(F.col("dst").alias("id")))
vertices_co = nodo(dim_p, "proveedor_key", "P|", "proveedor", ["nombre", "region", "es_consorcio"]).join(extremos.distinct(), "id")
grado_co = extremos.groupBy("id").count()
q_grado = grado_co.approxQuantile("count", [.5, .9, .99], .01)
print(f"co-participación: {vertices_co.count():,} nodos | {aristas_co.count():,} aristas | grado p50/p90/p99: {q_grado}")
(grado_co.orderBy(F.desc("count")).limit(5).join(vertices_co.select("id", "nombre"), "id").orderBy(F.desc("count"))
    .show(truncate=50))

co-participación: 60,213 nodos | 1,639,945 aristas | grado p50/p90/p99: [16.0, 113.0, 12059.0]


+-----------------+-----+---------------------------------------+
|               id|count|                                 nombre|
+-----------------+-----+---------------------------------------+
|P|RUC:20611499591|12059|              TVT QONTRATISTAS E.I.R.L.|
|P|RUC:20605753907| 9522|INVERSIONES GENERALES LAURA ROCA S.A.C.|
|P|RUC:20542513439| 7854|      CRISAMYR SOCIEDAD ANONIMA CERRADA|
|P|RUC:20612573621| 7168|               CONSTRUCAPRIGLE E.I.R.L.|
|P|RUC:20542059291| 6523|         CORPORACION GRUPO IBIZA S.A.C.|
+-----------------+-----+---------------------------------------+



- **Arista:** conectan si participaron juntos en **≥ 2 procesos** de entre 2 y `MAX_PART` participantes

- **Peso** = Σ 1/(n−1): coincidir en un proceso de 3 pesa 0.5 y en uno de 30, 0.03

- **Tope y umbral:** los masivos dominarían el grafo con vínculos débiles

- **Uso:** comunidades (Girvan-Newman/espectral en subgrafo, BigCLAM) y Node2Vec (PQ2)

---
### 4. Particionamiento y plan de ejecución

#### a. Self-join de co-participación

In [14]:
def tiempo(df):
    t0 = time.time(); n = df.count(); return n, round(time.time() - t0, 1)

part_sin = part.select("ocid", "proveedor_key").join(n_part.where(f"n BETWEEN 2 AND {MAX_PART}"), "ocid")
pares_sin = (part_sin.alias("a").join(part_sin.alias("b"),
                (F.col("a.ocid") == F.col("b.ocid")) & (F.col("a.proveedor_key") < F.col("b.proveedor_key"))))
pares_sin.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [ocid#82], [ocid#3946], Inner, (proveedor_key#83 < proveedor_key#3947)
   :- Project [ocid#82, proveedor_key#83, n#2542L]
   :  +- SortMergeJoin [ocid#82], [ocid#3936], Inner
   :     :- Sort [ocid#82 ASC NULLS FIRST], false, 0
   :     :  +- Exchange hashpartitioning(ocid#82, 24), ENSURE_REQUIREMENTS, [plan_id=2425]
   :     :     +- Project [ocid#82, proveedor_key#83]
   :     :        +- Filter (isnotnull(ocid#82) AND isnotnull(proveedor_key#83))
   :     :           +- FileScan parquet [ocid#82,proveedor_key#83,anio#86,mes#87] Batched: true, DataFilters: [isnotnull(ocid#82), isnotnull(proveedor_key#83)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/final/participantes], PartitionFilters: [anio#86 IN (2023,2024,2025)], PushedFilters: [IsNotNull(ocid), IsNotNull(proveedor_key)], ReadSchema: struct<ocid:string,proveedor_key:string>
   :     +- Sort [ocid#3936 ASC NULLS FIRST],

In [15]:
pares.explain()
print("sin repartition:", tiempo(pares_sin), "| con repartition(ocid) + cache:", tiempo(pares))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [proveedor_key#83 AS src#2601, proveedor_key#2575 AS dst#2602, n#2542L]
   +- SortMergeJoin [ocid#82], [ocid#2574], Inner, (proveedor_key#83 < proveedor_key#2575)
      :- Filter (isnotnull(ocid#82) AND isnotnull(proveedor_key#83))
      :  +- InMemoryTableScan [ocid#82, proveedor_key#83, n#2542L], [isnotnull(ocid#82), isnotnull(proveedor_key#83)]
      :        +- InMemoryRelation [ocid#82, proveedor_key#83, n#2542L], StorageLevel(disk, memory, deserialized, 1 replicas)
      :              +- *(5) Project [ocid#82, proveedor_key#83, n#2542L]
      :                 +- *(5) SortMergeJoin [ocid#82], [ocid#2547], Inner
      :                    :- *(2) Sort [ocid#82 ASC NULLS FIRST], false, 0
      :                    :  +- Exchange hashpartitioning(ocid#82, 24), ENSURE_REQUIREMENTS, [plan_id=2469]
      :                    :     +- *(1) Project [ocid#82, proveedor_key#83]
      :                    :        +- *(1) F

sin repartition: (13092446, 5.5) | con repartition(ocid) + cache: (13092446, 0.7)


- **Objetivo:** armar los pares de proveedores que participaron en el mismo proceso

- **Sin reparticionar:** `participantes` viene de disco particionada por `anio/mes`

- Participantes de un mismo proceso están en particiones distintas. 

- **Cada lado** del join vuelve a leer de disco y recalcula el conteo por proceso (con *shuffle*)

- **Approach 2 (`repartition(ocid)` + `cache`):** se reparten por `ocid`, y cada proceso queda entero en una partición

- **Tiempo:** 6.4 s → 0.8 s para los mismos 13.1M pares. Parte de la mejora es el cache (datos ya en memoria)

#### b. Broadcast y partition pruning

In [16]:
hechos.join(F.broadcast(dim_e), "entidad_key").explain()
leer(final_dir, "hechos").where("anio = 2025 AND mes = 3").select("ocid").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [entidad_key#3, ocid#0, award_id#1, fecha#2, proveedor_key#4, region#5, categoria#6, metodo_detalle#7, postor_unico#8, es_directa#9, n_postores#10, n_proveedores_adj#11L, tiene_contrato#12, monto_adj#13, monto_asignado#14, familia#15, segmento#16, anio#17, mes#18, nombre#96, ruc#97, region#98]
   +- BroadcastHashJoin [entidad_key#3], [entidad_key#95], Inner, BuildRight, false
      :- Filter isnotnull(entidad_key#3)
      :  +- FileScan parquet [ocid#0,award_id#1,fecha#2,entidad_key#3,proveedor_key#4,region#5,categoria#6,metodo_detalle#7,postor_unico#8,es_directa#9,n_postores#10,n_proveedores_adj#11L,tiene_contrato#12,monto_adj#13,monto_asignado#14,familia#15,segmento#16,anio#17,mes#18] Batched: true, DataFilters: [isnotnull(entidad_key#3)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/final/hechos], PartitionFilters: [anio#17 IN (2023,2024,2025)], PushedFilters: [IsNotNull(entidad_k

- **Broadcast join:** copia `entidades` a cada executor; `hechos` no se mueve
  - Agregar nombre y región a los hechos (armar los nodos del grafo)

- **Partition pruning:** con `anio = 2025 AND mes = 3`, lee solo esa carpeta de `hechos` (1 de 36 meses)
  
  - Separar años de forma barata

#### c. Sesgo

In [17]:
(aristas_co.groupBy(F.spark_partition_id().alias("particion")).count()
    .agg(F.min("count").alias("min"), F.expr("percentile_approx(count, 0.5)").alias("mediana"), F.max("count").alias("max")).show())

+-----+-------+-----+
|  min|mediana|  max|
+-----+-------+-----+
|67766|  68316|68907|
+-----+-------+-----+



- Las aristas de co-participación quedan **balanceadas** entre particiones (67,766 a 68,907 filas)

- El hash sobre `(src, dst)` reparte bien

- El sesgo está en los **nodos**, no en las particiones: unos pocos proveedores tienen grado > 12,000

---
### **5. Exportación de artefactos (`data/artefactos/`)**

In [18]:
artefactos = {
    "features_proveedor": feat_prov, "features_entidad": feat_ent,
    "matriz_entidad_proveedor": m_ep, "matriz_entidad_familia": m_ef,
    **{k: v.coalesce(1) for k, v in indices.items()},
    "grafo_bipartito/vertices": vertices_bip, "grafo_bipartito/aristas": aristas_bip,
    "grafo_coparticipacion/vertices": vertices_co, "grafo_coparticipacion/aristas": aristas_co}
for nombre, df in artefactos.items():
    df.write.mode("overwrite").parquet(str(artef_dir / nombre))
print("artefactos:", ", ".join(artefactos))

artefactos: features_proveedor, features_entidad, matriz_entidad_proveedor, matriz_entidad_familia, indice_entidad, indice_proveedor, indice_familia, grafo_bipartito/vertices, grafo_bipartito/aristas, grafo_coparticipacion/vertices, grafo_coparticipacion/aristas


| Artefacto | Grano | Técnica | Pregunta |
|---|---|---|---|
| `features_proveedor` | proveedor | KMeans++, DBSCAN/HDBSCAN, jerárquico | PQ1, PQ2 |
| `features_entidad` | entidad | ídem; entidades cautivas | PQ1 |
| `matriz_entidad_proveedor` | entidad × proveedor × año | ALS implícito, vecindario, SVD | PQ3 |
| `matriz_entidad_familia` | entidad × familia × año | perfil de rubros, *cold start* | PQ3 |
| `indice_*` | índice → clave | traducir resultados | — |
| `grafo_bipartito` | entidad → proveedor | PageRank, HITS | PQ1 |
| `grafo_coparticipacion` | proveedor — proveedor | comunidades, BigCLAM, Node2Vec | PQ2 |

### **6. Escritura de resultados**

In [19]:
resumen = pd.DataFrame([
    ("proveedores (features)", feat_prov.count()), ("entidades (features)", feat_ent.count()),
    ("pares entidad × proveedor", nnz_ep), ("densidad entidad × proveedor", d_ep),
    ("pares entidad × familia", nnz_ef), ("densidad entidad × familia", d_ef),
    ("vértices bipartito", vertices_bip.count()), ("aristas bipartito", aristas_bip.count()),
    ("vértices co-participación", vertices_co.count()), ("aristas co-participación", aristas_co.count())],
    columns=["medida", "valor"])
escritos = guardar_tablas({"resumen_artefactos": resumen,
                           "grado_coparticipacion": grado_co.groupBy(F.col("count").alias("grado")).agg(F.count("*").alias("nodos"))})
resumen.assign(valor=resumen.valor.map(lambda v: f"{v:.4%}" if v < 1 else f"{v:,.0f}"))

Escrito en 01_Trans/temp/c_uso/: resumen_artefactos, grado_coparticipacion


,medida,valor
0,proveedores (features),"86,017"
1,entidades (features),"2,996"
2,pares entidad × proveedor,"136,099"
3,densidad entidad × proveedor,0.0529%
4,pares entidad × familia,"39,348"
5,densidad entidad × familia,3.4070%
6,vértices bipartito,"89,009"
7,aristas bipartito,"136,099"
8,vértices co-participación,"60,213"
9,aristas co-participación,"1,639,945"


#### a. Verificación

In [20]:
verificar_salidas(escritos)

OK: 2 tablas (parquet + json)


---
### **7. Cerrar sesión**

In [21]:
spark.stop()